In [ ]:
%reset  -f
import mfem.ser as mfem
from glvis import glvis # visualización
import numpy as np


#### Difusión no lineal
$$\begin{array}{cl}-\nabla\cdot (\kappa(u)\nabla u) = f & \text{en } \Omega \\
u = 0 & \text{en }\partial \Omega\end{array}
$$
con $\kappa(u) = 1 + u^2$

#### Formulación Variacional no lineal:
$$\int_\Omega \kappa(u)\nabla u \nabla v - \int_\Omega fv = 0$$
que escribimos como
$$F(u) = 0$$
con 
$$F(u) = \int_\Omega \kappa(u)\nabla u \nabla v - \int_\Omega f v .$$
Usaremos el método de Newton para resolver, por lo que será preciso calcular
$$\langle \delta F(u), \delta u\rangle = \int_\Omega \kappa'(u)\nabla u\,\delta u \, \nabla v + \int_\Omega \kappa(u)\, \nabla \delta u\nabla v$$

In [ ]:
meshfile = 'mallas/circulo.mesh'

mesh = mfem.Mesh(meshfile)
dim = mesh.Dimension()
ref_levels = 1
for i in range(ref_levels):
    mesh.UniformRefinement()

In [ ]:
order = 1
fec = mfem.H1_FECollection(order, dim)

fespace = mfem.FiniteElementSpace(mesh, fec)
print("Número de incógnitas:",fespace.GetVSize())

In [ ]:
def pFunc(x):
    return 1.+x*x

def dpFunc(x):
    return 2*x

#### Coeficientes para el integrador no lineal

In [ ]:
# Coeficiente k(u)\nabla u
class NonlinearCoefficient(mfem.VectorPyCoefficientBase):
    def __init__(self, dim, gf):
        super(NonlinearCoefficient, self).__init__(dim,0)
        self.gf = gf
    def Eval(self, elvect, T, ip):        
        self.gf.GetGradient(T,elvect)
        val = self.gf.GetValue(T,ip)
        elvect *= pFunc(val)

# Coeficiente k(u)
class NonlinearDerivativeCoefficient(mfem.PyCoefficientBase):
    def __init__(self, gf):
        super(NonlinearDerivativeCoefficient, self).__init__(0) # inicialización en la clase padre
        self.gf = gf
    def Eval(self, T, ip):
        val = self.gf.GetValue(T,ip)
        return pFunc(val)

# Coeficiente k'(u)\nabla u
class NonlinearDerivativeCoefficient2(mfem.VectorPyCoefficientBase):
    def __init__(self, dim, gf):
        super(NonlinearDerivativeCoefficient2, self).__init__(dim,0) # inicialización en la clase padre
        self.gf = gf
    def Eval(self, elvect, T, ip):
        self.gf.GetGradient(T, elvect)
        val = self.gf.GetValue(T,ip)
        elvect *= dpFunc(val)


class rhsCoef(mfem.PyCoefficient):
    def EvalValue(self, x):
         return 12*x[0]**4 + 24*x[0]**2*x[1]**2 - 16*x[0]**2 +12*x[1]**4 - 16*x[1]**2 +8;
     

brhs = rhsCoef()

#### Integrador para la forma no lineal

In [ ]:
class NonlinearDiffusionIntegrator(mfem.PyNonlinearFormIntegrator):
    def __init__(self,fes):
        super().__init__()
        self.fes = fes
        self.gf = mfem.GridFunction(fes)

    # Corresponde a k(u)\nabla u \nabla v (integrador de la ecuación no lineal)
    def AssembleElementVector(self, el, T, elfun, elvect):
        dofs = self.fes.GetElementDofs(T.ElementNo)        
        ardofs = mfem.intArray(dofs)
        self.gf.SetSubVector(ardofs,elfun)
        
        coeff = NonlinearCoefficient(dim, self.gf)
        integ = mfem.DomainLFGradIntegrator(coeff)        
        integ.AssembleRHSElementVect(el, T, elvect)
        
        integ2 = mfem.DomainLFIntegrator(brhs)
        elvect2 = mfem.Vector()
        integ2.AssembleRHSElementVect(el, T, elvect2)
        elvect.Add(-1., elvect2)

    # Corresponde a la derivada
    def AssembleElementGrad(self, el, T, elfun, elmat):
        dofs = self.fes.GetElementDofs(T.ElementNo)        
        ardofs = mfem.intArray(dofs)
        
        self.gf.SetSubVector(ardofs, elfun)
        coeff = NonlinearDerivativeCoefficient(self.gf)
        integ1 = mfem.DiffusionIntegrator(coeff)
        integ1.AssembleElementMatrix(el,T,elmat)

        coeff2 = NonlinearDerivativeCoefficient2(dim,self.gf)
        integ2 = mfem.MixedScalarWeakDivergenceIntegrator(coeff2)
        elmat2 = mfem.DenseMatrix()
        integ2.AssembleElementMatrix(el,T,elmat2)
        
        elmat.Add(1.,elmat2)

#### Dato en la frontera

In [ ]:
x = mfem.GridFunction(fespace)
x.Assign(0.)


#### Crearmos la forma no lineal y añadimos nuestro integrador

In [ ]:
nform = mfem.NonlinearForm(fespace)
nonlin = NonlinearDiffusionIntegrator(fespace)
nform.AddDomainIntegrator(nonlin)

b = mfem.LinearForm(fespace)
b.Assign(0.)

# Frontera Dirichlet
nform.SetEssentialBC(mesh.bdr_attributes,b)

#### Solver lineal (para las iteraciones internas de Newton) y no lineal

In [ ]:
lsolver = mfem.CGSolver()
lsolver.SetMaxIter(300)
lsolver.SetPrintLevel(0)

# NewtonSolver
newton_solver = mfem.NewtonSolver()
newton_solver.SetOperator(nform)
newton_solver.SetSolver(lsolver)
newton_solver.SetPrintLevel(1)
newton_solver.SetRelTol(1e-7)

newton_solver.Mult(b, x)    

In [ ]:
glvis((mesh,x),keys="RRc")